In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

## Load Database

In [2]:
from pathlib import Path
DB_PATH = Path.cwd().parent / "database" / "car_sales.parquet"

# Read database
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)
df = pd.read_parquet(DB_PATH)
df.head()

,car_id,date,day,month,year,customer_name,gender,dealer_name,company,model,engine,transmission,color,dealer_no_,body_style,phone,dealer_region,price,income_customer,quantity,discount,gross_sales,discount_amount,sales,cost,total_cost,profit,profit_margin
0,C_CND_000001,2022-01-02,2,1,2022,Geraldine,Male,Buddy Storbeck's Diesel Service Inc,Ford,Expedition,DoubleÂ Overhead Camshaft,Auto,Black,06457-3834,SUV,8264678,Middletown,"467,740,000.00","242,865,000.00",1,0.05,"467,740,000.00","23,387,000.00","444,353,000.00","347,588,502.33","347,588,502.33","96,764,497.67",21.78
1,C_CND_000002,2022-01-02,2,1,2022,Gia,Male,C & M Motors Inc,Dodge,Durango,DoubleÂ Overhead Camshaft,Auto,Black,60504-7114,SUV,6848189,Aurora,"341,810,000.00","26,625,200,000.00",5,0.10,"1,709,050,000.00","170,905,000.00","1,538,145,000.00","281,435,978.53","1,407,179,892.65","130,965,107.35",8.51
2,C_CND_000003,2022-01-02,2,1,2022,Gianna,Male,Capitol KIA,Cadillac,Eldorado,Overhead Camshaft,Manual,Red,38701-8047,Passenger,7298798,Greenville,"566,685,000.00","18,619,650,000.00",2,0.02,"1,133,370,000.00","22,667,400.00","1,110,702,600.00","433,036,322.96","866,072,645.92","244,629,954.08",22.02
3,C_CND_000004,2022-01-02,2,1,2022,Giselle,Male,Chrysler of Tri-Cities,Toyota,Celica,Overhead Camshaft,Manual,Pale White,99301-3882,SUV,6257557,Pasco,"251,860,000.00","242,865,000.00",1,0.50,"251,860,000.00","125,930,000.00","125,930,000.00","190,288,699.80","190,288,699.80","-64,358,699.80",-51.11
4,C_CND_000005,2022-01-02,2,1,2022,Grace,Male,Chrysler Plymouth,Acura,TL,DoubleÂ Overhead Camshaft,Auto,Red,53546-9427,Hatchback,7081483,Janesville,"440,755,000.00","26,355,350,000.00",4,0.10,"1,763,020,000.00","176,302,000.00","1,586,718,000.00","371,011,737.64","1,484,046,950.56","102,671,049.44",6.47


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23905 entries, 0 to 23904
Data columns (total 28 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   car_id           23905 non-null  object 
 1   date             23905 non-null  object 
 2   day              23905 non-null  int32  
 3   month            23905 non-null  int32  
 4   year             23905 non-null  int32  
 5   customer_name    23905 non-null  object 
 6   gender           23905 non-null  object 
 7   dealer_name      23905 non-null  object 
 8   company          23905 non-null  object 
 9   model            23905 non-null  object 
 10  engine           23905 non-null  object 
 11  transmission     23905 non-null  object 
 12  color            23905 non-null  object 
 13  dealer_no_       23905 non-null  object 
 14  body_style       23905 non-null  object 
 15  phone            23905 non-null  int64  
 16  dealer_region    23905 non-null  object 
 17  price       

## Feature engineering

In [4]:
# Create column feature day_of_week, is_weekend, is_workday
df['date'] = pd.to_datetime(df['date'])
df['day_of_week'] = df['date'].dt.dayofweek
df['is_weekend'] = df['day_of_week'].apply(lambda x: 1 if x >= 5 else 0)
df['is_workday'] = df['day_of_week'].apply(lambda x: 1 if x < 5 else 0)

In [5]:
# Build Season feature to define the season of the year based on the month
def season(month):
    if month in [12, 1, 2]:
        return "Winter"
    elif month in [3, 4, 5]:
        return "Spring"
    elif month in [6, 7, 8]:
        return "Summer"
    else:
        return "Autumn"
    
df["season"] = df["month"].apply(season)

In [6]:
# Adjust date into dividen specific periods
df["quarter"] = df["date"].dt.quarter
df["week_of_year"] = df["date"].dt.isocalendar().week.astype(int)

In [7]:
# Build feature price_band as arrangement of price into 4 bands
df["price_band"] = pd.qcut(
    df["price"], q=5, labels=[
        "Budget","Economy", "Mid", "Premium", "Luxury"
    ]
)

In [8]:
# Create a feature of discount_level
df['discount_level'] = pd.cut(
    df['discount'], bins=[0,0.05,0.1,0.15,1],
    labels=[
        "Low",
        "Medium",
        "High",
        "Extreme"
    ]
)

# Create feature of weekend only discount by multiplying is_weekend and discount
df['weekend_discount'] = (df['is_weekend'] * df['discount'])

## Data clean

In [9]:
df.head()

,car_id,date,day,month,year,customer_name,gender,dealer_name,company,model,engine,transmission,color,dealer_no_,body_style,phone,dealer_region,price,income_customer,quantity,discount,gross_sales,discount_amount,sales,cost,total_cost,profit,profit_margin,day_of_week,is_weekend,is_workday,season,quarter,week_of_year,price_band,discount_level,weekend_discount
0,C_CND_000001,2022-01-02,2,1,2022,Geraldine,Male,Buddy Storbeck's Diesel Service Inc,Ford,Expedition,DoubleÂ Overhead Camshaft,Auto,Black,06457-3834,SUV,8264678,Middletown,"467,740,000.00","242,865,000.00",1,0.05,"467,740,000.00","23,387,000.00","444,353,000.00","347,588,502.33","347,588,502.33","96,764,497.67",21.78,6,1,0,Winter,1,52,Mid,Low,0.05
1,C_CND_000002,2022-01-02,2,1,2022,Gia,Male,C & M Motors Inc,Dodge,Durango,DoubleÂ Overhead Camshaft,Auto,Black,60504-7114,SUV,6848189,Aurora,"341,810,000.00","26,625,200,000.00",5,0.10,"1,709,050,000.00","170,905,000.00","1,538,145,000.00","281,435,978.53","1,407,179,892.65","130,965,107.35",8.51,6,1,0,Winter,1,52,Economy,Medium,0.10
2,C_CND_000003,2022-01-02,2,1,2022,Gianna,Male,Capitol KIA,Cadillac,Eldorado,Overhead Camshaft,Manual,Red,38701-8047,Passenger,7298798,Greenville,"566,685,000.00","18,619,650,000.00",2,0.02,"1,133,370,000.00","22,667,400.00","1,110,702,600.00","433,036,322.96","866,072,645.92","244,629,954.08",22.02,6,1,0,Winter,1,52,Premium,Low,0.02
3,C_CND_000004,2022-01-02,2,1,2022,Giselle,Male,Chrysler of Tri-Cities,Toyota,Celica,Overhead Camshaft,Manual,Pale White,99301-3882,SUV,6257557,Pasco,"251,860,000.00","242,865,000.00",1,0.50,"251,860,000.00","125,930,000.00","125,930,000.00","190,288,699.80","190,288,699.80","-64,358,699.80",-51.11,6,1,0,Winter,1,52,Budget,Extreme,0.50
4,C_CND_000005,2022-01-02,2,1,2022,Grace,Male,Chrysler Plymouth,Acura,TL,DoubleÂ Overhead Camshaft,Auto,Red,53546-9427,Hatchback,7081483,Janesville,"440,755,000.00","26,355,350,000.00",4,0.10,"1,763,020,000.00","176,302,000.00","1,586,718,000.00","371,011,737.64","1,484,046,950.56","102,671,049.44",6.47,6,1,0,Winter,1,52,Mid,Medium,0.10


## Data encoding

In [10]:
from sklearn.preprocessing import LabelEncoder

df_ml = df.copy()
le = LabelEncoder()

# Encode the object data in dataframe
for col in df_ml.select_dtypes(include=['object', 'category']).columns:
    df_ml[col] = le.fit_transform(df_ml[col])

In [11]:
df_ml.head(2)

,car_id,date,day,month,year,customer_name,gender,dealer_name,company,model,engine,transmission,color,dealer_no_,body_style,phone,dealer_region,price,income_customer,quantity,discount,gross_sales,discount_amount,sales,cost,total_cost,profit,profit_margin,day_of_week,is_weekend,is_workday,season,quarter,week_of_year,price_band,discount_level,weekend_discount
0,0,2022-01-02,2,1,2022,1050,1,0,8,60,0,0,0,0,3,8264678,4,"467,740,000.00","242,865,000.00",1,0.05,"467,740,000.00","23,387,000.00","444,353,000.00","347,588,502.33","347,588,502.33","96,764,497.67",21.78,6,1,0,3,1,52,3,1,0.05
1,1,2022-01-02,2,1,2022,1057,1,1,7,52,0,0,0,3,3,6848189,0,"341,810,000.00","26,625,200,000.00",5,0.10,"1,709,050,000.00","170,905,000.00","1,538,145,000.00","281,435,978.53","1,407,179,892.65","130,965,107.35",8.51,6,1,0,3,1,52,1,2,0.10


## Normalize Data Skewed

In [12]:
skewed_features = ["income_customer", "price", "sales", "gross_sales", "cost", "total_cost", "profit", "discount_amount"]
for feature in skewed_features:
    df_ml[feature] = np.log1p(df_ml[feature])
df_ml.head(2)

,car_id,date,day,month,year,customer_name,gender,dealer_name,company,model,engine,transmission,color,dealer_no_,body_style,phone,dealer_region,price,income_customer,quantity,discount,gross_sales,discount_amount,sales,cost,total_cost,profit,profit_margin,day_of_week,is_weekend,is_workday,season,quarter,week_of_year,price_band,discount_level,weekend_discount
0,0,2022-01-02,2,1,2022,1050,1,0,8,60,0,0,0,0,3,8264678,4,19.96,19.31,1,0.05,19.96,16.97,19.91,19.67,19.67,18.39,21.78,6,1,0,3,1,52,3,1,0.05
1,1,2022-01-02,2,1,2022,1057,1,1,7,52,0,0,0,3,3,6848189,0,19.65,24.01,5,0.10,21.26,18.96,21.15,19.46,21.06,18.69,8.51,6,1,0,3,1,52,1,2,0.10


## Split data into X and y

In [13]:
from sklearn.model_selection import train_test_split
X = df_ml[["gender","income_customer", "dealer_name", "dealer_region", "company", "model",
           "color", "body_style", "price", "discount", "day_of_week", "season", "week_of_year", "price_band", "sales"]]
y = df_ml["quantity"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")

X_train shape: (19124, 15), y_train shape: (19124,)


## Data scaling

In [14]:
from sklearn.preprocessing import StandardScaler
feature_scaler = StandardScaler()
target_scaler = StandardScaler()

X_train_scaled = pd.DataFrame(feature_scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(feature_scaler.transform(X_test), columns=X_test.columns, index=X_test.index)
y_train_scaled = target_scaler.fit_transform(y_train.values.reshape(-1, 1)).ravel()
y_test_scaled = target_scaler.transform(y_test.values.reshape(-1, 1)).ravel()

In [15]:
X_train_scaled.head(2)

,gender,income_customer,dealer_name,dealer_region,company,model,color,body_style,price,discount,day_of_week,season,week_of_year,price_band,sales
19155,0.52,0.84,0.88,-0.48,0.32,0.46,0.18,-1.61,-0.36,-0.69,1.40,-1.10,0.48,0.72,1.19
10018,0.52,0.88,-0.48,1.54,1.59,0.64,-1.22,-0.86,-2.04,-0.69,1.40,1.48,1.25,-1.38,-0.54


### Machine learning models - Demand prediction (Quantity as a target)

In [16]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, mean_absolute_percentage_error

models = {
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(random_state=42),
    "CatBoost": CatBoostRegressor(random_state=42, verbose=0),
    "XGBoost": XGBRegressor(random_state=42, verbosity=0, objective='reg:squarederror', tree_method='hist')
}

parameters = {
    "Decision Tree": {
        "max_depth": [None, 5, 10, 15],
        "min_samples_split": [2, 5, 10, 20],
        "min_samples_leaf": [1, 2, 4, 8]
    },
    "Random Forest": {
        "n_estimators": [100, 200, 300],
        "max_depth": [10, 15, 20, None],
        "min_samples_split": [2, 5, 10, 20],
        "min_samples_leaf": [1, 2, 4],
        "max_features": ["sqrt", "log2"]
    },
    "CatBoost": {
        "iterations": [200, 300],
        "depth": [4, 6, 8],
        "learning_rate": [0.01, 0.05, 0.1],
        "l2_leaf_reg": [1, 3, 5, 7]
    },
    "XGBoost": {
        "n_estimators": [200, 300],
        "max_depth": [4, 6, 8],
        "learning_rate": [0.01, 0.05, 0.1],
        "subsample": [0.8, 0.9, 1.0],
        "colsample_bytree": [0.8, 0.9, 1.0]
    }
}

# Train and evaluate models
trained_models = {}
results = []
feature_importances = {}
best_parameters = {}

for model_name, model in models.items():
    print("=" * 50)
    print(f"Training {model_name}...")
    print("=" * 50)

    # Machine learning models with hyperparameter tuning using RandomizedSearchCV
    random_search = RandomizedSearchCV(
        estimator=model,
        param_distributions=parameters[model_name],
        n_iter=10,
        cv=5,
        scoring='r2',
        n_jobs=-1,
        verbose=1,
        random_state=42
    )
    random_search.fit(X_train_scaled, y_train_scaled) # Train the model

    # Get the best model from RandomizedSearchCV
    best_model = random_search.best_estimator_
    
    # Get the best parameters and store them
    best_parameters[model_name] = random_search.best_params_
    print(f"Best parameters for {model_name}: {random_search.best_params_}")
    
    # Store the trained model
    trained_models[model_name] = best_model

    # Best parameters
    print("\nBest Parameters")
    print(random_search.best_params_)

    # Predict on the test set
    y_pred = best_model.predict(X_test_scaled)

    # Evaluate the model
    mse = mean_squared_error(y_test_scaled, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test_scaled, y_pred)
    mape = mean_absolute_percentage_error(y_test_scaled, y_pred)
    r2 = r2_score(y_test_scaled, y_pred)

    # Append results to the list
    results.append({
        "Model": model_name,
        "MSE": mse,
        "RMSE": rmse,
        "MAE": mae,
        "MAPE": mape,
        "R2": r2
    })

    print(f"{model_name} Evaluation Metrics:")
    print(f"Mean Squared Error (MSE): {mse:.3f}")
    print(f"Root Mean Squared Error (RMSE): {rmse:.3f}")
    print(f"Mean Absolute Error (MAE): {mae:.3f}")
    print(f"Mean Absolute Percentage Error (MAPE): {mape:.3f}")
    print(f"R-squared (R2): {r2:.3f}")

    # Feature importance
    if hasattr(best_model, 'feature_importances_'):
        importance = pd.DataFrame({
            "Feature": X_train.columns,
            "Importance": best_model.feature_importances_
        }).sort_values(by="Importance", ascending=False)
        feature_importances[model_name] = importance

# ==========================================================
# Comparison Table
# ==========================================================
df_comparison = pd.DataFrame(results).sort_values(by="R2", ascending=False).reset_index(drop=True)
print("\nModel Comparison:")
print(df_comparison)

# Feature importance for each model
for model_name, importance in feature_importances.items():
    print("\n")
    print("=" * 50)
    print(f"{model_name} Feature Importance:")
    print("=" * 50)
    print(importance.head(20))

Training Decision Tree...
Fitting 5 folds for each of 10 candidates, totalling 50 fits
Best parameters for Decision Tree: {'min_samples_split': 2, 'min_samples_leaf': 2, 'max_depth': 15}

Best Parameters
{'min_samples_split': 2, 'min_samples_leaf': 2, 'max_depth': 15}
Decision Tree Evaluation Metrics:
Mean Squared Error (MSE): 0.005
Root Mean Squared Error (RMSE): 0.072
Mean Absolute Error (MAE): 0.007
Mean Absolute Percentage Error (MAPE): 0.015
R-squared (R2): 0.995
Training Random Forest...
Fitting 5 folds for each of 10 candidates, totalling 50 fits
Best parameters for Random Forest: {'n_estimators': 200, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'log2', 'max_depth': 20}

Best Parameters
{'n_estimators': 200, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'log2', 'max_depth': 20}
Random Forest Evaluation Metrics:
Mean Squared Error (MSE): 0.052
Root Mean Squared Error (RMSE): 0.227
Mean Absolute Error (MAE): 0.153
Mean Absolute Percentage Error 

In [ ]:
# Model comparison to dataframe
df_comparison

,Model,MSE,RMSE,MAE,MAPE,R2
0,XGBoost,0.00,0.04,0.02,0.04,1.00
1,CatBoost,0.00,0.04,0.02,0.05,1.00
2,Decision Tree,0.01,0.07,0.01,0.02,0.99
3,Random Forest,0.05,0.23,0.15,0.28,0.95


In [ ]:
importance = (
    pd.DataFrame({
        "Feature": X_train.columns,
        "Importance": trained_models["Random Forest"].feature_importances_
    })
    .sort_values("Importance", ascending=False)
)

print(importance)

            Feature  Importance
14            sales        0.53
1   income_customer        0.22
8             price        0.09
13       price_band        0.04
9          discount        0.02
5             model        0.02
12     week_of_year        0.01
2       dealer_name        0.01
4           company        0.01
3     dealer_region        0.01
10      day_of_week        0.01
7        body_style        0.01
11           season        0.00
6             color        0.00
0            gender        0.00


# Implement models prediction to database

In [ ]:
df_quantity_prediction = df.copy()

# Restore original quantity values
df_quantity_prediction["quantity"] = np.expm1(df_quantity_prediction["quantity"])

# Store model performance summary
model_summary = []

# Scale ALL feature using training scaler
X_all_scaled = pd.DataFrame(feature_scaler.transform(X), columns=X.columns, index=X.index)

# original target values
y_true = np.expm1(y)

for model_name, model in trained_models.items():
    print("=" * 70)
    print(f"Predicting Quantity using {model_name}")
    print("=" * 70)

    # Predict on scaled data
    y_pred_scaled = model.predict(X_all_scaled)

    # Convert back to original quantity scale
    y_pred_log = target_scaler.inverse_transform(y_pred_scaled.reshape(-1, 1)).flatten()

    # Back to original quantity scale
    y_pred = np.expm1(y_pred_log)

    # Save prediction
    df_quantity_prediction[f"{model_name}_predicted_quantity"] = np.round(y_pred, 2)

    # Row-level error analysis
    df_quantity_prediction[f"{model_name}_absolute_error"] = np.abs(y_true - y_pred)
    df_quantity_prediction[f"{model_name}_squared_error"] = (y_true - y_pred) ** 2

    # Overall Model Metrics
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    model_summary.append({
        "Model": model_name,
        "MSE": mse,
        "RMSE": rmse,
        "MAE": mae,
        "MAPE": mape,
        "R2": r2
    })

    print(f"{model_name} Overall Performance Metrics:")
    print(f"Mean Squared Error (MSE): {mse:.3f}")
    print(f"Root Mean Squared Error (RMSE): {rmse:.3f}")
    print(f"Mean Absolute Error (MAE): {mae:.3f}")
    print(f"Mean Absolute Percentage Error (MAPE): {mape:.3f}")
    print(f"R-squared (R2): {r2:.3f}")

Predicting Quantity using Decision Tree
Decision Tree Overall Performance Metrics:
Mean Squared Error (MSE): 7.863
Root Mean Squared Error (RMSE): 2.804
Mean Absolute Error (MAE): 0.146
Mean Absolute Percentage Error (MAPE): 0.004
R-squared (R2): 0.996
Predicting Quantity using Random Forest
Random Forest Overall Performance Metrics:
Mean Squared Error (MSE): 190.742
Root Mean Squared Error (RMSE): 13.811
Mean Absolute Error (MAE): 4.860
Mean Absolute Percentage Error (MAPE): 0.124
R-squared (R2): 0.900
Predicting Quantity using CatBoost
CatBoost Overall Performance Metrics:
Mean Squared Error (MSE): 6.879
Root Mean Squared Error (RMSE): 2.623
Mean Absolute Error (MAE): 0.925
Mean Absolute Percentage Error (MAPE): 0.029
R-squared (R2): 0.996
Predicting Quantity using XGBoost
XGBoost Overall Performance Metrics:
Mean Squared Error (MSE): 4.438
Root Mean Squared Error (RMSE): 2.107
Mean Absolute Error (MAE): 0.623
Mean Absolute Percentage Error (MAPE): 0.020
R-squared (R2): 0.998


In [ ]:
# Model performance summary
df_model_summary = pd.DataFrame(model_summary).sort_values(by="R2", ascending=False).reset_index(drop=True)
print("\n")
print("=" * 70)
print("MODEL PERFORMANCE SUMMARY")
print("=" * 70)

print(df_model_summary)

# Prediction columns
prediction_columns = [
    "quantity",
    "Decision Tree_predicted_quantity",
    "Random Forest_predicted_quantity",
    "CatBoost_predicted_quantity",
    "XGBoost_predicted_quantity"
]

print("\nPrediction Preview")
display(df_quantity_prediction[prediction_columns].head(20))



MODEL PERFORMANCE SUMMARY
           Model    MSE  RMSE  MAE  MAPE   R2
0        XGBoost   4.44  2.11 0.62  0.02 1.00
1       CatBoost   6.88  2.62 0.92  0.03 1.00
2  Decision Tree   7.86  2.80 0.15  0.00 1.00
3  Random Forest 190.74 13.81 4.86  0.12 0.90

Prediction Preview


,quantity,Decision Tree_predicted_quantity,Random Forest_predicted_quantity,CatBoost_predicted_quantity,XGBoost_predicted_quantity
0,1.72,1.72,1.87,1.72,1.67
1,147.41,147.41,117.06,147.94,149.57
2,6.39,6.39,7.79,7.24,6.56
3,1.72,1.72,1.95,1.76,1.76
4,53.60,53.60,55.68,55.09,55.74
5,147.41,147.41,126.48,146.72,146.49
6,6.39,6.39,7.60,6.84,6.61
7,1.72,1.72,1.91,1.72,1.66
8,147.41,147.41,97.79,132.48,142.27
9,6.39,6.39,6.21,6.32,6.23


In [ ]:
df_quantity_prediction.drop(columns=[
    "Decision Tree_predicted_quantity",
    "Random Forest_predicted_quantity",
    "CatBoost_predicted_quantity",
    "XGBoost_predicted_quantity",
    "Decision Tree_absolute_error",
    "Random Forest_absolute_error",
    "CatBoost_absolute_error",
    "XGBoost_absolute_error",
    "Decision Tree_squared_error",
    "Random Forest_squared_error",
    "CatBoost_squared_error",
    "XGBoost_squared_error"
], inplace=True)

In [ ]:
df_quantity_prediction

,car_id,date,day,month,year,customer_name,gender,dealer_name,company,model,engine,transmission,color,dealer_no_,body_style,phone,dealer_region,price,income_customer,quantity,discount,gross_sales,discount_amount,sales,cost,total_cost,profit,profit_margin,day_of_week,is_weekend,is_workday,season,quarter,week_of_year,price_band,discount_level,weekend_discount
0,C_CND_000001,2022-01-02,2,1,2022,Geraldine,Male,Buddy Storbeck's Diesel Service Inc,Ford,Expedition,DoubleÂ Overhead Camshaft,Auto,Black,06457-3834,SUV,8264678,Middletown,"467,740,000.00","242,865,000.00",1.72,0.05,"467,740,000.00","23,387,000.00","444,353,000.00","347,588,502.33","347,588,502.33","96,764,497.67",21.78,6,1,0,Winter,1,52,Mid,Low,0.05
1,C_CND_000002,2022-01-02,2,1,2022,Gia,Male,C & M Motors Inc,Dodge,Durango,DoubleÂ Overhead Camshaft,Auto,Black,60504-7114,SUV,6848189,Aurora,"341,810,000.00","26,625,200,000.00",147.41,0.10,"1,709,050,000.00","170,905,000.00","1,538,145,000.00","281,435,978.53","1,407,179,892.65","130,965,107.35",8.51,6,1,0,Winter,1,52,Economy,Medium,0.10
2,C_CND_000003,2022-01-02,2,1,2022,Gianna,Male,Capitol KIA,Cadillac,Eldorado,Overhead Camshaft,Manual,Red,38701-8047,Passenger,7298798,Greenville,"566,685,000.00","18,619,650,000.00",6.39,0.02,"1,133,370,000.00","22,667,400.00","1,110,702,600.00","433,036,322.96","866,072,645.92","244,629,954.08",22.02,6,1,0,Winter,1,52,Premium,Low,0.02
3,C_CND_000004,2022-01-02,2,1,2022,Giselle,Male,Chrysler of Tri-Cities,Toyota,Celica,Overhead Camshaft,Manual,Pale White,99301-3882,SUV,6257557,Pasco,"251,860,000.00","242,865,000.00",1.72,0.50,"251,860,000.00","125,930,000.00","125,930,000.00","190,288,699.80","190,288,699.80","-64,358,699.80",-51.11,6,1,0,Winter,1,52,Budget,Extreme,0.50
4,C_CND_000005,2022-01-02,2,1,2022,Grace,Male,Chrysler Plymouth,Acura,TL,DoubleÂ Overhead Camshaft,Auto,Red,53546-9427,Hatchback,7081483,Janesville,"440,755,000.00","26,355,350,000.00",53.60,0.10,"1,763,020,000.00","176,302,000.00","1,586,718,000.00","371,011,737.64","1,484,046,950.56","102,671,049.44",6.47,6,1,0,Winter,1,52,Mid,Medium,0.10
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23900,C_CND_023902,2023-12-31,31,12,2023,Martin,Male,C & M Motors Inc,Plymouth,Voyager,Overhead Camshaft,Manual,Red,60504-7114,Passenger,8583598,Pasco,"215,880,000.00","242,865,000.00",6.39,0.50,"431,760,000.00","215,880,000.00","215,880,000.00","159,542,978.96","319,085,957.92","-103,205,957.92",-47.81,6,1,0,Winter,4,52,Budget,Extreme,0.50
23901,C_CND_023903,2023-12-31,31,12,2023,Jimmy,Female,Ryder Truck Rental and Leasing,Chevrolet,Prizm,DoubleÂ Overhead Camshaft,Auto,Black,06457-3834,Hardtop,7914229,Middletown,"287,840,000.00","16,191,000,000.00",147.41,0.25,"1,439,200,000.00","359,800,000.00","1,079,400,000.00","202,866,442.05","1,014,332,210.25","65,067,789.75",6.03,6,1,0,Winter,4,52,Budget,Extreme,0.25
23902,C_CND_023904,2023-12-31,31,12,2023,Emma,Male,Chrysler of Tri-Cities,BMW,328i,Overhead Camshaft,Manual,Red,99301-3882,Sedan,7659127,Scottsdale,"377,790,000.00","12,682,950,000.00",1.72,0.50,"377,790,000.00","188,895,000.00","188,895,000.00","311,393,568.00","311,393,568.00","-122,498,568.00",-64.85,6,1,0,Winter,4,52,Economy,Extreme,0.50
23903,C_CND_023905,2023-12-31,31,12,2023,Victoire,Male,Chrysler Plymouth,Chevrolet,Metro,DoubleÂ Overhead Camshaft,Auto,Black,53546-9427,Passenger,6030764,Austin,"557,690,000.00","242,865,000.00",1.72,0.10,"557,690,000.00","55,769,000.00","501,921,000.00","438,804,951.00","438,804,951.00","63,116,049.00",12.57,6,1,0,Winter,4,52,Premium,Medium,0.10


In [ ]:
df_quantity_prediction['company']

0             Ford
1            Dodge
2         Cadillac
3           Toyota
4            Acura
           ...    
23900     Plymouth
23901    Chevrolet
23902          BMW
23903    Chevrolet
23904        Lexus
Name: company, Length: 23905, dtype: object

## Save models

In [ ]:
from pathlib import Path
#import joblib
import json

BASE_DIR = Path.cwd().parent / "models"
MODEL_DIR = BASE_DIR / "quantity_prediction"

#folders = [
    "models",
    "metrics",
    "parameters",
    "scalers",
    "encoders",
    "feature_importance"
]

for folder in folders:
    (MODEL_DIR / folder).mkdir(parents=True, exist_ok=True)

IndentationError: unexpected indent (3889202950.py, line 9)

In [ ]:
for model_name, model in trained_models.items():
    filename = model_name.lower().replace(" ", "_")

    if model_name == "CatBoost":
        model.save_model(MODEL_DIR / "models" / f"{filename}.cbm")
    elif model_name == "XGBoost":
        model.save_model(MODEL_DIR / "models" / f"{filename}.json")
    else:
        joblib.dump(model, MODEL_DIR / "models" / f"{filename}.pkl")

print("\nAll models have been saved successfully.")


All models have been saved successfully.


## Save scalers

In [ ]:
joblib.dump(feature_scaler, MODEL_DIR / "scalers" / "feature_scaler.pkl")
joblib.dump(target_scaler, MODEL_DIR / "scalers" / "target_scaler.pkl")

print("\nFeature and target scalers have been saved successfully.")


Feature and target scalers have been saved successfully.


## Save encders

In [ ]:
# Create encoders based on this cell of dataset what i've been processed
df_encoders = df.copy()

encoders = {}
for col in df.select_dtypes(include=['object', 'category']).columns:
    le = LabelEncoder()
    df_encoders[col] = le.fit_transform(df_encoders[col])
    encoders[col] = le

In [ ]:
encoders

{'car_id': LabelEncoder(),
 'customer_name': LabelEncoder(),
 'gender': LabelEncoder(),
 'dealer_name': LabelEncoder(),
 'company': LabelEncoder(),
 'model': LabelEncoder(),
 'engine': LabelEncoder(),
 'transmission': LabelEncoder(),
 'color': LabelEncoder(),
 'dealer_no_': LabelEncoder(),
 'body_style': LabelEncoder(),
 'dealer_region': LabelEncoder(),
 'season': LabelEncoder(),
 'price_band': LabelEncoder(),
 'discount_level': LabelEncoder()}

In [ ]:
joblib.dump(
    encoders,
    MODEL_DIR /
    "encoders" /
    "label_encoders.pkl"
)

print("Encoders saved.")

Encoders saved.


## Save metrics

In [ ]:
df_model_summary.to_csv(MODEL_DIR / "metrics" / "model_metrics.csv", index=False)
df_comparison.to_csv(MODEL_DIR / "metrics" / "model_comparison.csv", index=False)

print("\nModel metrics and comparison have been saved successfully.")


Model metrics and comparison have been saved successfully.


## Save feature importance

In [ ]:
# Save feature importance to csv
for model_name, importance in feature_importances.items():
    clean_name = model_name.lower().replace(" ", "_")
    importance.to_csv(MODEL_DIR / "feature_importance" / f"{clean_name}_feature_importance.csv", index=False)

## Save feature names

In [ ]:
with open(MODEL_DIR / "parameters" / "feature_columns.json", "w") as f:
    json.dump(X.columns.tolist(), f, indent=4)

print("\nFeature columns have been saved successfully.")


Feature columns have been saved successfully.
